# Get events

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "msc_thesis").is_dir()
)
sys.path.insert(0, str(project_root / "src"))

events_dir = project_root / "data" / "processed" / "events"
EVENTS_CSV_PATH = events_dir / "setlistfm_events.csv"
ENRICHMENTS_CSV_PATH = events_dir / "setlistfm_event_enrichments.csv"

from msc_thesis.data.events import SetlistFMEventSource

load_dotenv()

API_KEY = os.getenv("SETLISTFM_API_KEY")

In [2]:
source = SetlistFMEventSource(api_key=API_KEY, sleep_seconds=1)

data = source.fetch_events(
    country_code="DK",
    state="Capital Region",
    year=2019,
    per_page=20,
)

print(f"Total setlists fetched: {len(data.get('setlist', []))}")

Total setlists fetched: 1509


In [3]:
from msc_thesis.data.events import events_from_setlists, save_events_dataframe

events_df = events_from_setlists(data)
saved_path = save_events_dataframe(events_df, EVENTS_CSV_PATH)

print(f"Saved events dataframe to {saved_path}")
print(events_df.shape)

Saved events dataframe to /Users/aleksandrawozniak/Desktop/Thesis/msc-thesis/data/processed/events/setlistfm_events.csv
(1509, 21)


In [4]:
display(events_df)

,event_name,event_date,start_time,end_time,artist_name,artist_mbid,artist_url,tour_name,venue_name,venue_id,...,city,country,country_code,latitude,longitude,notes,setlist_id,setlist_url,last_updated,enriched
0,Siamese @ High Voltage,31-12-2019,None,None,Siamese,4d4760b1-f4de-4792-b63d-895e3f8163ad,https://www.setlist.fm/setlists/siamese-6bdb42...,NaN,High Voltage,bd4b18e,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,439b17df,https://www.setlist.fm/setlist/siamese/2019/hi...,2023-04-03T10:25:28.296+0000,False
1,DR SymfoniOrkestret @ DR Koncerthuset,30-12-2019,None,None,DR SymfoniOrkestret,0957da35-a754-4e00-9222-436dc0e4a388,https://www.setlist.fm/setlists/dr-symfoniorke...,Nytårsgalla,DR Koncerthuset,23d32cff,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,6bbad6e2,https://www.setlist.fm/setlist/dr-symfoniorkes...,2023-02-25T18:14:56.545+0000,False
2,Magtens Korridorer @ KB Hallen,28-12-2019,None,None,Magtens Korridorer,c8b3d0c4-c9c5-469f-abdd-312337aa3380,https://www.setlist.fm/setlists/magtens-korrid...,NaN,KB Hallen,53d62399,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,39bd97b,https://www.setlist.fm/setlist/magtens-korrido...,2019-12-26T13:34:25.425+0000,False
3,Alkymist @ Loppen,27-12-2019,None,None,Alkymist,e97e50a3-6310-4045-90d1-066707d2e0ed,https://www.setlist.fm/setlists/alkymist-43cbf...,NaN,Loppen,4bd6d34e,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,b9bd57a,https://www.setlist.fm/setlist/alkymist/2019/l...,2019-12-27T00:10:18.195+0000,False
4,Besvärjelsen @ Loppen,27-12-2019,None,None,Besvärjelsen,b611904b-b972-4e30-8861-11b1d7f3f48f,https://www.setlist.fm/setlists/besvarjelsen-3...,NaN,Loppen,4bd6d34e,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,7b9bda80,https://www.setlist.fm/setlist/besvarjelsen/20...,2019-12-28T00:55:13.224+0000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1504,Judith Hill @ Studie 2,11-01-2019,None,None,Judith Hill,14ea9fb0-132b-4377-8f21-0e849f6bd7de,https://www.setlist.fm/setlists/judith-hill-3b...,NaN,Studie 2,23d26017,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,7b950e14,https://www.setlist.fm/setlist/judith-hill/201...,2019-02-09T21:44:22.463+0000,False
1505,Architects @ Store Vega,08-01-2019,None,None,Architects,05dffdbe-dc6e-4c8d-a075-50a09c4cb45c,https://www.setlist.fm/setlists/architects-13d...,Holy Hell,Store Vega,73d61eb9,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,4b94075e,https://www.setlist.fm/setlist/architects/2019...,2019-01-09T12:08:19.883+0000,False
1506,Beartooth @ Store Vega,08-01-2019,None,None,Beartooth,98a1e0ab-35fa-40dd-b62c-9fda46fdb061,https://www.setlist.fm/setlists/beartooth-2bda...,NaN,Store Vega,73d61eb9,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,3b9474ec,https://www.setlist.fm/setlist/beartooth/2019/...,2019-01-09T23:34:17.855+0000,False
1507,Polaris @ Store VEGA,08-01-2019,None,None,Polaris,6ffb7be4-0a53-4014-853d-83790cd5e091,https://www.setlist.fm/setlists/polaris-1bc6ed...,NaN,Store VEGA,73d61eb9,...,Copenhagen,Denmark,DK,55.677681,12.570934,NaN,2b9474e6,https://www.setlist.fm/setlist/polaris/2019/st...,2020-12-26T20:57:35.097+0000,False


In [5]:
events_df.shape

(1509, 21)

# Get LLM output

In [9]:
import importlib
import msc_thesis.data.events.enrichment as event_enrichment

# Refresh the source module if it changed during this notebook session.
importlib.reload(event_enrichment)
EventEnricher = event_enrichment.EventEnricher

GEMINI_MODELS = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
]
enricher = EventEnricher(models=GEMINI_MODELS, max_retries=3)

In [10]:
import pandas as pd

events_df = pd.read_csv(EVENTS_CSV_PATH)
if "enriched" not in events_df.columns:
    events_df["enriched"] = False

events_df["enriched"] = events_df["enriched"].map(EventEnricher._as_bool)
pending_events_df = events_df.loc[~events_df["enriched"]].head(20)

print(f"Enriching {len(pending_events_df)} pending events.")

enriched_batch_df = enricher.enrich_dataframe(
    pending_events_df,
    source_csv_path=EVENTS_CSV_PATH,
    enriched_output_path=ENRICHMENTS_CSV_PATH,
    sleep_seconds=1,
)

display(enriched_batch_df)

Enriching 20 pending events.
Gemini gemini-3.6-flash request failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 2s (1/3).
[1/20] Enriched: 39bd57b
Gemini gemini-3.6-flash request failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 2s (1/3).
Gemini gemini-3.6-flash request failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}); retrying in 4s (2/3).
Gemini gemini-3.6-flash request failed (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usual

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}